In [15]:
import matplotlib.pyplot as plt
import pymupdf4llm
import pymupdf
import glob
import sys
import os
import re
import pandas as pd
import tqdm

sys.path.append("../src")
import text_extraction
import fitz
import tqdm
import os
import glob
import pandas as pd
import pymupdf


In [3]:
reports = glob.glob("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/*.pdf")
reports

['/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Deutsche Lufthansa AG1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Hermes International SCA2.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/RWE AG1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Nordea Bank Abp1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Knorr-Bremse AG1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Rolls-Royce Holdings plc1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/BAE Systems plc1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/

In [4]:
text_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/german_annual_reports_no_tables"
text_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/german_annual_reports_text_one_font_size"
text_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/german_annual_reports_text_heuristik"
text_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/german_annual_reports_text_combi"

In [5]:
def is_block_within(block1, block2):
    """
    Returns True if block1 is fully inside block2's bounding box.
    Each block must have a 'bbox' key: [x0, y0, x1, y1]
    """
    x0_1, y0_1, x1_1, y1_1 = block1
    x0_2, y0_2, x1_2, y1_2 = block2
    
    return (
        x0_1 >= x0_2 and
        y0_1 >= y0_2 and
        x0_1 <= x1_2 and
        y0_1 <= y1_2
    )

In [6]:
def get_paragraphs(pdf_path: str):

    doc = pymupdf.open(pdf_path)
    uniform_blocks = []
    temp_text = ""
    temp_font_size = 0

    for i, page in enumerate(doc): 

        if i == 50: 
            break

        blocks = page.get_text("dict")["blocks"]
        for block in blocks:
            if "lines" in block.keys(): 
                for line in block["lines"]: 
                    for span in line["spans"]:
                        if span["text"].strip():
                            if span["size"] == temp_font_size:
                                temp_text += span["text"]
                            else:
                                if temp_text.strip() and any(char.isalpha() for char in temp_text):
                                    temp_dict = {"text": temp_text, "page": page.number, "font_size": temp_font_size}
                                    uniform_blocks.append(temp_dict)
                                temp_text = span["text"]
                                temp_font_size = span["size"]
    df = pd.DataFrame(uniform_blocks)

    return df


In [7]:
def get_paragraphs_max_font_size(pdf_path: str):

    doc = pymupdf.open(pdf_path)
    uniform_blocks = []
    temp_text = ""
    temp_font_size = 0

    for page in doc:
        blocks = page.get_text("dict")["blocks"]
        for block in blocks:
            if "lines" in block.keys(): 
                for line in block["lines"]: 
                    for span in line["spans"]:
                        if span["text"].strip():
                            if span["size"] == temp_font_size:
                                temp_text += span["text"]
                            else:
                                if temp_text.strip() and any(char.isalpha() for char in temp_text):
                                    temp_dict = {"text": temp_text, "page": page.number, "font_size": temp_font_size}
                                    uniform_blocks.append(temp_dict)
                                temp_text = span["text"]
                                temp_font_size = span["size"]
    df = pd.DataFrame(uniform_blocks)

    return df


In [8]:
def get_paragraphs_without_table(pdf_path: str) -> pd.DataFrame:

    doc = pymupdf.open(pdf_path)
    uniform_blocks = []
    temp_text = ""
    temp_font_size = 0

    for i, page in tqdm.tqdm(enumerate(doc)): 
            
        tabs = page.find_tables()
        blocks = page.get_text("dict")["blocks"]
        
        if i == 50: 
            break

        for block in blocks:
            if block["type"] == 0:  # Nur Textblöcke (nicht Bilder)
                if not any([is_block_within(block["bbox"], tab.bbox) for tab in tabs]):             
                    if "lines" in block.keys(): 
                        for line in block["lines"]: 
                            for span in line["spans"]:
                                if span["text"].strip(): 
                                    if span["size"] == temp_font_size:
                                        temp_text += span["text"]
                                    else:
                                        if temp_text.strip() and any(char.isalpha() for char in temp_text):
                                            temp_dict = {"text": temp_text, "page": page.number, "font_size": temp_font_size}
                                            uniform_blocks.append(temp_dict)
                                        temp_text = span["text"]
                                        temp_font_size = span["size"]

    df = pd.DataFrame(uniform_blocks)

    df["len"] = df["text"].apply(len)
    df["share"] = df["len"] / df["len"].sum()    
    return df

In [9]:
def get_font_sizes(path): 
    df = get_paragraphs_without_table(path)
    df["len"] = df["text"].apply(len)
    df["share"] = df["len"] / df["len"].sum()    
    return df.groupby("font_size").sum().sort_values(by="share", ascending=False)

In [10]:
def extract_all_sizes(block):
    sizes = []
    for line in block.get("lines", []):
        for span in line.get("spans", []):
            if "size" in span:
                sizes.append(span["size"])
    return sizes

In [11]:
def extract_block_text(block):
    spans = []
    for line in block.get("lines", []):
        for span in line.get("spans", []):
            if span["text"].strip():
                spans.append(span["text"])
    text = " ".join(spans)
    text = re.sub(r' +', ' ', text)
    return " ".join(spans)

In [12]:
def heuristik_text(text: str) -> bool: 
    # remove all numbers like 1.234 or 500.000
    text = re.sub(r'\b\d+\.\d+\b', '', text)

    text = re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", "", text)
    
    if len(text.split(" ")) > 10 and len(text) > 30 and "." in text: 
            return True
    
    return False

In [47]:
def check_font_sizes(max_font_size: float, font_sizes_block: list): 
    for font_size_block in font_sizes_block: 
        if font_size_block < max_font_size * 1.02 and font_size_block > max_font_size * 0.98: 
            return True

In [48]:
def check_nummerical_block(text: str, p: float = 0.5):
    text = text.strip()

    if len(text) == 0: 
        return False

    count = sum(c.isdigit() for c in text)
    return count/len(text) > p

In [49]:
reports

['/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Deutsche Lufthansa AG1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Hermes International SCA2.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/RWE AG1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Nordea Bank Abp1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Knorr-Bremse AG1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Rolls-Royce Holdings plc1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/BAE Systems plc1.pdf',
 '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/

In [50]:
import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io

# --- Helper function ---
def is_block_within(block_bbox, table_bbox):
    bx0, by0, bx1, by1 = block_bbox
    tx0, ty0, tx1, ty1 = table_bbox
    return bx0 >= tx0 and by0 >= ty0 and bx1 <= tx1 and by1 <= ty1

# --- Load PDF ---

record_font_sizes = {}

for path in tqdm.tqdm(reports):
    doc = fitz.open(path)

    df = get_paragraphs(path)
    df["len"] = df["text"].apply(len)
    df["share"] = df["len"] / df["len"].sum()    
    font_sizes =  df.groupby("font_size").sum().sort_values(by="share", ascending=False)
    record_font_sizes[os.path.basename(path)] = font_sizes
    max_font_size =  font_sizes.iloc[0].name

    images = []

    i = 0

    # --- Iterate over all pages ---
    for page in doc.pages():
        
        i += 1
        if i == 50:
            break

        # Get image of the page
        pix = page.get_pixmap(dpi=150)
        img_bytes = pix.tobytes("ppm")
        img = Image.open(io.BytesIO(img_bytes))

        # Setup figure and axes
        fig, ax = plt.subplots(figsize=(10, 12))
        ax.imshow(img)
        ax.axis("off")
        
        # DPI scaling factor
        scale = 150 / 72

        # Table and block detection
        tables = page.find_tables()
        blocks = page.get_text("dict")["blocks"]

        # Draw block bounding boxes
        for b in blocks:
            if b["type"] == 0:
                
                x0, y0, x1, y1 = b["bbox"]

                text_block = extract_block_text(b)

                font_sizes_block = extract_all_sizes(b)

                if any([is_block_within(b["bbox"], tab.bbox) for tab in tables]): 
                    color = "red"
                elif check_nummerical_block(text_block): 
                    color = "brown"
                elif check_font_sizes(max_font_size=max_font_size, font_sizes_block=font_sizes_block):
                    color = "yellow"
                elif heuristik_text(text_block): 
                    color = "blue"
                else: 
                    color = "black"

                rect = patches.Rectangle(
                    (x0 * scale, y0 * scale), (x1 - x0) * scale, (y1 - y0) * scale,
                    linewidth=1.5, edgecolor=color, facecolor='none'
                )
                ax.add_patch(rect)

        # Save figure to in-memory image
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', dpi=150)
        buf.seek(0)
        images.append(Image.open(buf).convert("RGB"))
        plt.close(fig)

    # --- Save all images into a single PDF ---

    save_path = os.path.join(text_path, os.path.basename(path))

    if images:
        images[0].save(save_path, save_all=True, append_images=images[1:])

    print(f"Saved {save_path} successfully.")

  0%|          | 1/275 [00:27<2:07:26, 27.91s/it]

Saved /Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/german_annual_reports_text_combi/Deutsche Lufthansa AG1.pdf successfully.


  1%|          | 2/275 [00:50<1:53:36, 24.97s/it]

Saved /Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/german_annual_reports_text_combi/Hermes International SCA2.pdf successfully.


  1%|          | 3/275 [01:12<1:46:55, 23.59s/it]

...

 47%|████▋     | 130/275 [59:26<1:02:42, 25.95s/it]

Saved /Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/german_annual_reports_text_combi/Ipsen SA2.pdf successfully.


 48%|████▊     | 131/275 [59:48<59:27, 24.77s/it]  

Saved /Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/german_annual_reports_text_combi/Stellantis N.V.1.pdf successfully.


: 

In [14]:
heuristik_text("€m1,509– 1,666EBIT€m1,231– 2,316Net profit/loss€m791– 2,191Key balance sheet and cash flow statement figuresTotal assets€m43,33542,5382Equity ratio%19.610.69.0 ptsNet indebtedness€m6,8719,023– 24Pension provisions€m2,0696,676– 69Operating cash flow ")

True

In [ ]:
for r in record_font_sizes: 
    plt.bar(r.index, r["share"])
    plt.show()

In [42]:
def get_text_spans_without_tables(pdf_path: str) -> pd.DataFrame:

    doc = pymupdf.open(pdf_path)
    uniform_blocks = []

    block_nr = 0
    i = 0
    for page in doc:
        i += 1
        if i == 10: 
            break
        blocks = page.get_text("dict")["blocks"]
        tables = page.find_tables()
        print(i)
        for tab in tables: 
            print(tab.to_pandas())
        for block in blocks:
            if not any([is_block_within(block["bbox"], tab.bbox) for tab in tables]) and "lines" in block.keys(): 
                for line in block["lines"]: 
                    for span in line["spans"]:
                        span_text = ""
                        span_font_size = 0
                        if span["text"].strip():
                            span_text += span["text"]
                            span_font_size = span["size"]
                        if span_text.strip():                                        
                            temp_dict = {"text": span_text, "page": page.number, "font_size": span_font_size, "block_nr": block_nr}
                            uniform_blocks.append(temp_dict)
            block_nr += 1
        
    df = pd.DataFrame(uniform_blocks)

    return df


In [129]:
reports[0] = '/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/Lotus Bakeries NV1.pdf'

In [130]:
df_spans = get_text_spans_without_tables(reports[0])

1
2
3
4
5
6
7
8
9


In [131]:
df_spans["len"] = df_spans["text"].apply(len)
df_spans["share"] = df_spans["len"] / df_spans["len"].sum()    
font_sizes =  df_spans.groupby("font_size").sum().sort_values(by="share", ascending=False)
max_font_size =  font_sizes.iloc[0].name

In [132]:
blocks = df_spans.groupby("block_nr").agg({
    "page": lambda x: x.mean(), 
    "text" : lambda x: " ".join(x), 
    "font_size": lambda x: x.to_list(),
})

In [133]:
blocks

,page,text,font_size
block_nr,,,
0,0.0,ANNUAL REPORT 2022,[20.0]
3,1.0,OUR MISSION IS TO CREATE SMALL MOMENTS OF JOY ...,"[20.0, 20.0, 20.0, 20.0, 20.0, 20.0]"
4,2.0,Lotus Bakeries - 5,[10.0]
5,2.0,A MESSAGE FROM OUR CEO AND CHAIRMAN,"[28.0, 28.0]"
6,2.0,It is not outlandish to say that the entire wo...,[7.971750259399414]
...,...,...,...
241,8.0,"stake in IQBAR, an American producer of",[8.0]
242,8.0,a plant-based nutrition bars containing six,[8.0]
243,8.0,ingredients with benefits for the body and,[8.0]


In [134]:
import re

def is_block_within(block_bbox, table_bbox):
    bx0, by0, bx1, by1 = block_bbox
    tx0, ty0, tx1, ty1 = table_bbox
    return bx0 >= tx0 and by0 >= ty0 and bx1 <= tx1 and by1 <= ty1

def extract_all_sizes(block):
    sizes = []
    for line in block.get("lines", []):
        for span in line.get("spans", []):
            if "size" in span:
                sizes.append(span["size"])
    return sizes

def extract_block_text(block):
    spans = []
    for line in block.get("lines", []):
        for span in line.get("spans", []):
            if span["text"].strip():
                spans.append(span["text"])
    text = " ".join(spans)
    text = re.sub(r' +', ' ', text)
    return " ".join(spans)

def heuristik_text(text: str) -> bool: 
    # remove all numbers like 1.234 or 500.000
    text = re.sub(r'\b\d+\.\d+\b', '', text)

    text = re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", "", text)
    
    if len(text.split(" ")) > 10 and len(text) > 30 and "." in text: 
            return True
    
    return False
def check_font_sizes(max_font_size: float, font_sizes_block: list): 
    for font_size_block in font_sizes_block: 
        if font_size_block < max_font_size * 1.02 and font_size_block > max_font_size * 0.98: 
            return True
    return False

def check_nummerical_block(text: str, p: float = 0.5):
    text = text.strip()

    if len(text) == 0: 
        return False

    count = sum(c.isdigit() for c in text)
    return count/len(text) > p

In [135]:
## filter blocks
blocks["accept"] = None
for idx, block in blocks.iterrows(): 
    blocks.loc[idx, "accept"] = not check_nummerical_block(block["text"]) and (check_font_sizes(max_font_size, block["font_size"]) or heuristik_text(block["text"]))

In [136]:
blocks

,page,text,font_size,accept
block_nr,,,,
0,0.0,ANNUAL REPORT 2022,[20.0],False
3,1.0,OUR MISSION IS TO CREATE SMALL MOMENTS OF JOY ...,"[20.0, 20.0, 20.0, 20.0, 20.0, 20.0]",True
4,2.0,Lotus Bakeries - 5,[10.0],False
5,2.0,A MESSAGE FROM OUR CEO AND CHAIRMAN,"[28.0, 28.0]",False
6,2.0,It is not outlandish to say that the entire wo...,[7.971750259399414],True
...,...,...,...,...
241,8.0,"stake in IQBAR, an American producer of",[8.0],True
242,8.0,a plant-based nutrition bars containing six,[8.0],True
243,8.0,ingredients with benefits for the body and,[8.0],True


In [137]:
# join (take other example pdf!)d

In [138]:
import copy

In [139]:
blocks["font_size_set"] = blocks["font_size"].apply(set)

In [140]:
blocks_2 = []

for page in set(blocks["page"]): 
    blocks_temp = copy.copy(blocks[blocks["page"] == page])
    
    
    for i, row in blocks_temp.iterrows(): 
        if row["accept"] and len(row["font_size_set"]) == 1: 
            temp_text = row["text"]
            temp_font_size = list(row["font_size_set"])[0]
            start_index = i
            break

    for i, row in blocks_temp.loc[start_index:].iterrows(): 
        if row["accept"] and len(row["font_size_set"]) == 1 and temp_font_size == list(row["font_size_set"])[0]: 
            temp_text += row["text"]
        elif row["accept"] and len(row["font_size_set"]) > 1: 
            blocks_2.append({"text": temp_text, "font_size": temp_font_size})
            temp_text = ""
            temp_font_size = 
        else: 
            blocks_2.append({"text": temp_text, "font_size": temp_font_size})

SyntaxError: invalid syntax (4034761042.py, line 20)

In [141]:
start_index

174

In [142]:
blocks_temp.iloc[0]["font_size_set"]

{9.0}

In [ ]:
blocks_temp["font_size_set"] = blocks_temp["font_size"].apply(set)

'|   block_nr |   page | text                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [118]:
print(blocks[30:40].to_markdown())

|   block_nr |   page | text                                                                                                                                                                                                                  | font_size                 | accept   | font_size_set            |
|-----------:|-------:|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:--------------------------|:---------|:-------------------------|
|         33 |      1 | Last page viewed                                                                                                                                                                                                      | [9.0]                     | True     | {9.0}                    |
|         34 |      1 | Internet   references                                     

In [143]:

import pandas as pd

def merge_runs(df,
               text_col="text",
               font_set_col="font_size_set",
               accept_col="accept",
               keep_cols=("row_id", "page")):
    """
    Merge consecutive rows that have:
      • accept == True
      • len(font_size_set) == 1
      • identical font_size_set values
    Concatenates the text of such runs (separated by a single space) and
    retains the first values encountered for all other columns
    listed in `keep_cols`.
    """
    if df.empty:
        return df

    # Make sure the DataFrame is in the original order
    df = df.reset_index(drop=True)

    merged_rows = []          # collect collapsed records here
    buffer = df.iloc[0].copy()  # row we are currently building

    for _, row in df.iloc[1:].iterrows():
        can_merge = (
            buffer[accept_col]                               # both rows accepted …
            and row[accept_col]
            and len(buffer[font_set_col]) == 1               # … each has one font size …
            and len(row[font_set_col]) == 1
            and buffer[font_set_col] == row[font_set_col]    # … and the size is the same
        )

        if can_merge:
            # Glue the texts together
            buffer[text_col] = f"{buffer[text_col]} {row[text_col]}"
            # Optional: accumulate the raw font-size list if you need it
            if "font_sizes" in df.columns:
                buffer["font_sizes"] = buffer.get("font_sizes", []) + row["font_sizes"]
        else:
            merged_rows.append(buffer)
            buffer = row.copy()         # start a new run

    merged_rows.append(buffer)          # add the final buffered record
    result = pd.DataFrame(merged_rows)

    # Keep only desired columns plus the ones we modified/need
    wanted = list(keep_cols) + [text_col, "font_sizes", font_set_col, accept_col]
    return result[[c for c in wanted if c in result.columns]].reset_index(drop=True)

In [144]:
blocks_merged = merge_runs(blocks)

In [1]:
blocks_merged

NameError: name 'blocks_merged' is not defined